# CAM · Curso de estrategia
Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso. No escribas la contraseña en una celda de código.

Después ejecuta los ejercicios en orden. El código editable y los notebooks se preparan automáticamente en **Archivos → curso-motorsport**. Haz doble clic en un archivo `.py` para leerlo y editarlo dentro de Colab. No necesitas descargar ni subir nada manualmente. Guarda una copia del notebook en Drive; los cambios en los archivos Python de la sesión son temporales.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Monte Carlo: cuándo activar el Attack Mode

Compararemos varias estrategias (vueltas de activación) de un mismo piloto. La parrilla es siempre la misma.

La carrera tiene 25 vueltas. Hay una sola activación de Attack Mode, con una duración de 4 vueltas y da 2 adelantamientos por vuelta. Cada uno de nuestros rivales activa de manera aleatoria entre la vuelta 3 y la 21.

## Inicialización

Lo primero que haremos es importar las librerías relevantes.

In [ ]:
# Librerías propias (ver src/race_models/fe)
from race_models.fe.config import RaceConfig
from race_models.fe.monte_carlo import run_monte_carlo
from race_models.fe.analysis.mc_plots import plot_monte_carlo_exp, plot_monte_carlo_dist, plot_monte_carlo_distribution
from race_models.fe.analysis.mc_tables import monte_carlo_dataframe
from race_models.fe.analysis.race_plots import plot_race

# Librerías de Python
import pandas as pd
import numpy as np
from IPython.display import display

Ahora iniciaremos la configuración de la carrera. En este caso, utilizaremos los valores por defecto.

In [ ]:
race_config = RaceConfig()

Además, definiremos una lista que contiene cada una de las estrategias que queremos validar con la Monte Carlo, definidas según su vuelta de activación.

En este caso, probaremos todas las estrategias de activación entre la vuelta 3 y la 25.

In [ ]:
activation_plans = [(lap,) for lap in range(3, race_config.total_laps + 1)]  # Lista de estrategias, definidas según su vuelta de activación

activation_plans

# Parte 1 - Pascal Wehrlein, pole

En esta primera parte, estudiaremos las opciones estratégicas desde el punto de vista de Wehrlein, que sale primero.

## Monte Carlo - Simulación

La función `run_monte_carlo` acepta:
- `race_config`: la configuración de la carrera (define parámetros como el número de vueltas, la longitud de los Attack Mode, etc.).
- `starting_position`: la posición de salida del coche al cual queremos aplicar las estrategias.
- `activation_plans`: la lista de estrategias.
- `simulations`: el número de carreras que correremos por estrategia.
- `sc`: define si los Safety Car están activos o no. Cuando lo están, aparecen de manera aleatoria.

In [ ]:
result = run_monte_carlo(
    race_config=race_config,
    starting_position=1,
    activation_plans=activation_plans,
    simulations=2500,
    sc=True
)

print(f"Carreras simuladas: {len(result.strategies) * len(result.scenario_seeds):,}")

## Visualización de los resultados


### Distribuciones

Todos los resultados posibles y sus probabilidades estimadas.

En la siguiente celda, visualizamos las distribuciones para las estrategias con vuelta de activación 3, 10 y 20.

In [ ]:
plot_monte_carlo_dist(result, activation_lap=[3, 10, 20]);

### Valores esperados y desviaciones estándar

- Valor esperado: media de los resultados.
- Desviación estándar (STD): mide cuánto se dispersan los resultados alrededor de su media. Está asociada al riesgo de cada estrategia.

Calculamos los dos valores tanto para puntos como para posición final.

In [ ]:
df = monte_carlo_dataframe(result)
with pd.option_context('display.max_rows', None, 'display.float_format', '{:.3f}'.format):
    display(df)

In [ ]:
plot_monte_carlo_exp(result);

## Ver una carrera individual
Los índices empiezan en cero. `strategy_index` sigue el orden de `activation_plans`, no el de la tabla ordenada. El mismo `race_index` utiliza los mismos rivales en todas las estrategias. La carrera se reconstruye con la semilla guardada, incluso si ejecutaste la Monte Carlo con `seed=None`.

In [ ]:
activation_lap = 21

race_result = result.get_race(np.random.randint(0, 1000), strategy_index=activation_lap - 3)
plot_race(race_result, title=f"Monte Carlo - activación L{activation_lap}");

# Parte 2 - Pepe Martí

Ejercicio: repite el mismo análisis pero desde el punto de vista de Pepe Martí, decimotercero en parrilla.

¿Qué diferencias hay? ¿Por qué ocurren estas diferencias? ¿Cuál es la mejor estrategia?

In [ ]:
result = run_monte_carlo(
    race_config=race_config,
    starting_position=13,  # Pepe Martí sale decimotercero
    activation_plans=activation_plans,
    simulations=2500,
    sc=True
)

print(f"Carreras simuladas: {len(result.strategies) * len(result.scenario_seeds):,}")